In [1]:
!pip install catboost --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.9 MB/s eta 0:00:00


In [2]:
# Здесь готовлю себе json
from google.colab import drive
drive.mount('/content/drive')
source_dir = '/content/drive/MyDrive/2025_09_LCT/data/source'
data_dir = dest_dir = '/content/drive/MyDrive/2025_09_LCT/data/auto'
RANDOM_STATE = 42
#sentences_3629_processed.csv

import pandas as pd


Mounted at /content/drive


In [3]:
!ls {source_dir}

 200_labeled_reviews_new_02.10.2025_v3.txt   categories.json
'600_labeled_reviews_02.10.2025 v4.txt'      last_300.xlsx
 answ_0_4.txt				     my_xlm_roberta_model
 batch_1a7200fe_120202.json		     review_links_banki-ru_25.09.json
 batch_f4a39c30_114343.json		     reviews_banki-ru_25-09.csv
 batch_f4a39c30_114431.json		     reviews_banki-ru_25-09.json
 categories_25.09.csv			     sample.json
 categories_25.09.json			     sentences_498_processed.csv
 categories.csv				     sentences.csv


In [4]:
import os
import pandas as pd
import json

drive.mount('/content/drive')

!ls {dest_dir}


JSON_PATH = os.path.join(source_dir, 'sample.json')

'''
def safe_load_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Файл {file_path} не найден")
        return None
    except json.JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        return None
    except Exception as e:
        print(f"Неизвестная ошибка: {e}")
        return None

def load_json_files(names):
    """
    Загружает все JSON файлы из указанной папки
    """
    all_data = []

    for filename in names:
        #if filename.endswith('.json'):
            file_path = os.path.join(source_dir, filename)
            try:
                #with open(file_path, 'r', encoding='utf-8') as f:
                    #data = json.load(f)
                    data = safe_load_json(file_path)
                    all_data.extend(data)
                    #all_data.append(data)
            except Exception as e:
                print(f"Ошибка при загрузке файла {filename}: {e}")

    return all_data
data = load_json_files(['200_labeled_reviews_new_02.10.2025_v3.txt', '600_labeled_reviews_02.10.2025 v4.txt'])
#data = load_json_files(['600_labeled_reviews_02.10.2025 v4.txt'])
len(data)
labeled_id = [int(rev['id']) for rev in data]
type(labeled_id[0])
import pandas as pd
tmp_data = pd.read_csv(os.path.join(source_dir, 'reviews_banki-ru_25-09.csv'))

tmp_data[~tmp_data['id'].isin(labeled_id)]['review_text'].sample(250).to_json(JSON_PATH)
'''

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
100ep_full_data_model.pth		category_classifier_18_classes.cbm
18_cat_classifier_full_data.cbm		category_classifier_30_classes.cbm
251004_cat_classifier_30_classes.cbm	category_classifier.cbm
251004_sent_classifier_3_classes.cbm	joint_classification_model.cbm
80ep_85perc_data_model_fixed.pth	sent_classifier_3_classes.cbm
86ep_85perc_data_model_fixed_50_50.pth	sentences_1296_processed.csv
batch_1f1e4fd8_083151.json		sentences_1639_processed.csv
batch_371d859b_093151.json		sentences_1962_processed.csv
batch_4e5cd182_115229.json		sentences_2297_processed.csv
batch_60e237d8_112621.json		sentences_2622_processed.csv
batch_65f0ccbd_084912.json		sentences_2969_processed.csv
batch_91614183_081309.json		sentences_314_processed.csv
batch_aab1f84c_092526.json		sentences_3282_processed.csv
batch_e57f8250_081819.json		sentences_3629_processed.csv
batch_f13a2f63_090317.json

'\ndef safe_load_json(file_path):\n    try:\n        with open(file_path, \'r\', encoding=\'utf-8\') as f:\n            return json.load(f)\n    except FileNotFoundError:\n        print(f"Файл {file_path} не найден")\n        return None\n    except json.JSONDecodeError as e:\n        print(f"Ошибка парсинга JSON: {e}")\n        return None\n    except Exception as e:\n        print(f"Неизвестная ошибка: {e}")\n        return None\n\ndef load_json_files(names):\n    """\n    Загружает все JSON файлы из указанной папки\n    """\n    all_data = []\n\n    for filename in names:\n        #if filename.endswith(\'.json\'):\n            file_path = os.path.join(source_dir, filename)\n            try:\n                #with open(file_path, \'r\', encoding=\'utf-8\') as f:\n                    #data = json.load(f)\n                    data = safe_load_json(file_path)\n                    all_data.extend(data)\n                    #all_data.append(data)\n            except Exception as e:\n     

In [5]:
import json
import pandas as pd
from pathlib import Path
import re
#from torch.cuda import is_available

import os
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier, Pool

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Optional

RANDOM_STATE = RANDOM_SEED = 42
device = DEVICE = 'CPU' # 'GPU' if is_available() else 'CPU'

CATEGORIES_D = {
    1: "Дебетовые карты",
    2: "Кредитные карты",
    3: "Вклады",
    4: "Накопительные счета",
    5: "Потребительские кредиты",
    6: "Ипотека",
    7: "Страхование",
    8: "Денежные переводы, СБП",
    9: "Интернет-банк, мобильный банк и приложение",
    10: "Премиум-обслуживание",
    11: "Работа колл-центра и клиентского сервиса",
    12: "Безопасность и защита",
    13: "Общее впечатление о банке",
    14: "Кэшбэк, промокоды, бонусы, акции, приведи друга",
    15: "Прочие банковские услуги и сервисы",
    16: "Сравнение с конкурентами",
    17: "Тарифы, комиссии, прозрачность условий",
    18: "Эскалация и угроза жалоб"
}

# Обратный маппинг для поиска id по названию
CATEGORIES_REVERSE = {v: k for k, v in CATEGORIES_D.items()}

NUM_CLASSES = len(CATEGORIES_D)
NUM_SENT_CLASSES = 3

CAT_CLASSIFIER_PATH = os.path.join(dest_dir,
                                   f'18_cat_classifier_full_data.cbm')

SENT_CLASSIFIER_PATH = os.path.join(dest_dir,
                                   f'sent_classifier_3_classes.cbm')

MAX_SEQ_LEN = 50

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [6]:
with open(JSON_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)
data['2577']
#Если индексов не будет - нужно сгенерировать уникальные

'увидела на сайте газпромбанка акцию можно вернуть комиссию 5000 за выпуск карты union pay при условии, если остаток не снижается ниже 50 000 в течении 3х мес. подробностей не нашла - на сайте не было ссылки на полные правила, не было часто задаваемых вопросов. оформила карту 7 августа. но 7 и 8 августа у меня не было возможности зайти на сайт приложения у газпромбанка для айфона нет, а сайт с телефона было открывать проблемно. 9 августа я с ноутбука пополнила счет на 55 000 и с меня списали комиссию. неснижаемый остаток я поддерживала до ноября. ожидала, что мне вернут 5000 руб по истечению 90 дней, но этого не произошло. оказалось, что я должна была пополнить карту 7 или 8 августа. при этом по новым условиям можно пополнить в течении 6 дней с момента получения  на сайт добавили подробные условия, которых не было раньше. итог очень обидно, что из-за того, что газпромбанк не опубликовал полные условия на сайте, не опубликовал ссылку и я не догадалась сразу написать в техподдержку и спр

In [7]:
# Аббревиатуры
ABBREVIATIONS = {
    "ул.", "г.", "д.", "т.д.", "т.п.", "т.е.", "и т.д.", "и т.п.", "см.", "мин.",
    "руб.", "коп.", "рис.", "стр.", "обл.", "р-н", "просп.", "пер.", "бул.",
    "реф.", "тех.", "эл.", "смс", "др.", "т.к."
}

DATE_PATTERN = re.compile(r"\b\d{1,2}\.\d{1,2}\.\d{2,4}\b")


# Защита дат
def protect_dates(text: str):
    dates = DATE_PATTERN.findall(text)
    for i, d in enumerate(dates):
        text = text.replace(d, f"§DATE{i}§", 1)
    return text, dates

def restore_dates(text: str, dates):
    for i, d in enumerate(dates):
        text = text.replace(f"§DATE{i}§", d, 1)
    return text


# Базовая очистка
def clean_text(text: str) -> str:
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


# Основной цикл деления
def split_sentences(text: str):

    text = clean_text(text)
    text, dates = protect_dates(text)

    # Разделяем по знакам окончания предложения: ., !, ?
    # Скобки ) и » учитываем только как конец, если за ними пробел + заглавная буква или конец строки
    parts = re.split(r'(?<=[.!?])\s+', text)

    sentences = []
    for part in parts:
        if not part:
            continue
        if sentences:
            prev = sentences[-1]
            # Если предыдущее заканчивается на аббревиатуру или точку + маленькая буква, то объединяем
            if prev.endswith(tuple(ABBREVIATIONS)) or (
                prev.endswith(".") and part and part[0].islower()
            ):
                sentences[-1] = prev + " " + part
                continue
            # Если предыдущее заканчивается на ) и следующее с маленькой буквы, то объединяем
            if prev.endswith(")") and part and part[0].islower():
                sentences[-1] = prev + " " + part
                continue

        sentences.append(part)

    sentences = [restore_dates(s, dates).strip() for s in sentences if s.strip()]
    return sentences


sentences_data = []
for review_id, review_text in data.items():
        review_text = review_text
        sentences = split_sentences(review_text)

        for sentence_id, sentence in enumerate(sentences):
            sentences_data.append({
                'sentence_id': f"{review_id}_{sentence_id:03d}",  #
                'review_id': review_id,
                'sentence_text': sentence,
            })

tr_val_df = pd.DataFrame(sentences_data)
tr_val_df.head()

,sentence_id,review_id,sentence_text
0,2577_000,2577,"увидела на сайте газпромбанка акцию можно вернуть комиссию 5000 за выпуск карты union pay при условии, если остаток не снижается ниже 50 000 в течении 3х мес. подробностей не нашла - на сайте не было ссылки на полные правила, не было часто задаваемых вопросов. оформила карту 7 августа. но 7 и 8 августа у меня не было возможности зайти на сайт приложения у газпромбанка для айфона нет, а сайт с телефона было открывать проблемно."
1,2577_001,2577,"9 августа я с ноутбука пополнила счет на 55 000 и с меня списали комиссию. неснижаемый остаток я поддерживала до ноября. ожидала, что мне вернут 5000 руб по истечению 90 дней, но этого не произошло. оказалось, что я должна была пополнить карту 7 или 8 августа. при этом по новым условиям можно пополнить в течении 6 дней с момента получения на сайт добавили подробные условия, которых не было раньше. итог очень обидно, что из-за того, что газпромбанк не опубликовал полные условия на сайте, не опубликовал ссылку и я не догадалась сразу написать в техподдержку и спросить про условия акции я лишилась 5000. искать полные условия на сайте - тот еще квест. а еще обиднее, т.к. картой так и не получилось воспользоваться. в турции не смогла оплатить в ресторане заблокировали деньги на счете до тех пор, пока я не вернулась в россию. писала отзыв с негативной оценкой, но решили, что я обманута законно. в итоге отзыв не засчитали."
2,19971_000,19971,добрый день!
3,19971_001,19971,сегодня 23.10 оформила заявку на дебетовую карту. в настоящее время идёт рекламная кампания у банка.заполнила на сайте банка данные. через несколько минут после оформления заявки поступил звонок от мошенников. ко мне обратились по имени и отчеству!!!
4,19971_002,19971,какую ещё информацию получили посторонние лица?


In [8]:
cat_classifier = CatBoostClassifier()
cat_classifier.load_model(CAT_CLASSIFIER_PATH)

sent_classifier = CatBoostClassifier()
sent_classifier.load_model(SENT_CLASSIFIER_PATH)


In [9]:
#all_probs = cat_classifier.predict_proba(val_df['sentence_text'].tolist())
def add_probs(df, cat_classifier, sent_classifier):
    pool = Pool(df['sentence_text'].values, text_features=[0])
    cat_probs = cat_classifier.predict_proba(pool)
    cat_prob_columns = [f'cat_prob_cat_{i+1}' for i in range(cat_probs.shape[1])]
    df[cat_prob_columns] = cat_probs
    sent_probs = sent_classifier.predict_proba(pool)
    sent_prob_columns = [f'sent_prob_cat_{i+1}' for i in range(sent_probs.shape[1])]
    df[sent_prob_columns] = sent_probs

add_probs(tr_val_df, cat_classifier, sent_classifier)
tr_val_df.columns

Index(['sentence_id', 'review_id', 'sentence_text', 'cat_prob_cat_1',
       'cat_prob_cat_2', 'cat_prob_cat_3', 'cat_prob_cat_4', 'cat_prob_cat_5',
       'cat_prob_cat_6', 'cat_prob_cat_7', 'cat_prob_cat_8', 'cat_prob_cat_9',
       'cat_prob_cat_10', 'cat_prob_cat_11', 'cat_prob_cat_12',
       'cat_prob_cat_13', 'cat_prob_cat_14', 'cat_prob_cat_15',
       'cat_prob_cat_16', 'cat_prob_cat_17', 'cat_prob_cat_18',
       'sent_prob_cat_1', 'sent_prob_cat_2', 'sent_prob_cat_3'],
      dtype='object')

In [10]:
class TestReviewDataset(Dataset):
    def __init__(self, sentences_df,
                 num_categories=NUM_CLASSES,
                 num_sentiments=NUM_SENT_CLASSES,
                 max_seq_len=MAX_SEQ_LEN):
        self.sentences_df = sentences_df
        self.num_categories = num_categories
        self.num_sentiments = num_sentiments
        self.max_seq_len = max_seq_len

        # Группируем предложения по отзывам
        self.review_groups = sentences_df.groupby('review_id')
        self.review_ids = list(self.review_groups.groups.keys())

    def __len__(self):
        return len(self.review_ids)

    def __getitem__(self, idx):
        review_id = self.review_ids[idx]
        group = self.review_groups.get_group(review_id)

        # Берем вероятности категорий и сентиментов
        cat_prob_cols = [f'cat_prob_cat_{i+1}' for i in range(self.num_categories)]
        sent_prob_cols = [f'sent_prob_cat_{i+1}' for i in range(self.num_sentiments)]

        cat_probs = group[cat_prob_cols].values.astype(np.float32)
        sent_probs = group[sent_prob_cols].values.astype(np.float32)

        # Объединяем фичи
        sentence_features = np.concatenate([cat_probs, sent_probs], axis=1)

        # Паддинг
        if len(sentence_features) > self.max_seq_len:
            sentence_features = sentence_features[:self.max_seq_len]
        else:
            padding = np.zeros((self.max_seq_len - len(sentence_features),
                              self.num_categories + self.num_sentiments), dtype=np.float32)
            sentence_features = np.vstack([sentence_features, padding])

        return torch.FloatTensor(sentence_features), review_id

# Создание тестового датасета и лоадера
test_dataset = TestReviewDataset(tr_val_df)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
class ReviewCNN1D(nn.Module):
    def __init__(self,
                 num_categories=NUM_CLASSES,
                 num_sentiments=NUM_SENT_CLASSES,
                 hidden_dim=128,
                 num_filters=64,
                 kernel_size=3,
                 dropout=0.3):

        super().__init__()

        self.num_categories = num_categories
        self.num_sentiments = num_sentiments

        # Input: [seq_len, num_categories + num_sentiments] = [seq_len, 33]
        input_dim = num_categories + num_sentiments  # 30 + 3 = 33

        self.conv1d = nn.Sequential(
            nn.Conv1d(input_dim, num_filters, kernel_size, padding=kernel_size//2),  # [33, 64, 3]
            nn.BatchNorm1d(num_filters),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)

        combined_features = num_filters * 2

        self.category_head = nn.Sequential(
            nn.Linear(combined_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_categories),
            nn.Sigmoid()
        )

        self.sentiment_head = nn.Sequential(
            nn.Linear(combined_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_categories * num_sentiments)
        )

    def forward(self, x):
        # x: [batch_size, seq_len, 33] -> [batch_size, 33, seq_len]
        x = x.transpose(1, 2)
        x = self.conv1d(x)
        avg_pool = self.global_avg_pool(x).squeeze(-1)
        max_pool = self.global_max_pool(x).squeeze(-1)
        combined = torch.cat([avg_pool, max_pool], dim=1)

        category_probs = self.category_head(combined)
        sentiment_logits = self.sentiment_head(combined)
        sentiment_probs = F.softmax(
            sentiment_logits.view(-1, self.num_categories, self.num_sentiments),
            dim=-1
        )

        return category_probs, sentiment_probs

In [12]:
!ls {dest_dir}

100ep_full_data_model.pth		category_classifier_18_classes.cbm
18_cat_classifier_full_data.cbm		category_classifier_30_classes.cbm
251004_cat_classifier_30_classes.cbm	category_classifier.cbm
251004_sent_classifier_3_classes.cbm	joint_classification_model.cbm
80ep_85perc_data_model_fixed.pth	sent_classifier_3_classes.cbm
86ep_85perc_data_model_fixed_50_50.pth	sentences_1296_processed.csv
batch_1f1e4fd8_083151.json		sentences_1639_processed.csv
batch_371d859b_093151.json		sentences_1962_processed.csv
batch_4e5cd182_115229.json		sentences_2297_processed.csv
batch_60e237d8_112621.json		sentences_2622_processed.csv
batch_65f0ccbd_084912.json		sentences_2969_processed.csv
batch_91614183_081309.json		sentences_314_processed.csv
batch_aab1f84c_092526.json		sentences_3282_processed.csv
batch_e57f8250_081819.json		sentences_3629_processed.csv
batch_f13a2f63_090317.json		sentences_656_processed.csv
batch_f68a2733_114044.json		sentences_955_processed.csv
batch_f858725c_104318.json


In [13]:
model = ReviewCNN1D(
    num_categories=NUM_CLASSES,
    num_sentiments=NUM_SENT_CLASSES,
)

#torch.save(model.state_dict(), os.path.join(dest_dir, CONV_))
#model_name = '100ep_full_data_model.pth'
model_name = '86ep_85perc_data_model_fixed_50_50.pth'
# torch.save(model.state_dict(), os.path.join(dest_dir, model_name))
model.load_state_dict(torch.load(os.path.join(dest_dir, model_name)))

<All keys matched successfully>

In [14]:
def predict_reviews(model, test_loader, threshold=0.35):
    model.eval()
    all_predictions = []

    with torch.no_grad():
        for batch_x, batch_review_ids in test_loader:
            cat_probs, sent_probs = model(batch_x)

            # Преобразуем в numpy
            cat_probs = cat_probs.numpy()
            sent_probs = sent_probs.numpy()

            for i, review_id in enumerate(batch_review_ids):
                # Предсказанные категории (выше порога)
                pred_cat_indices = np.where(cat_probs[i] > threshold)[0]

                # ЕСЛИ НЕТ КАТЕГОРИЙ ВЫШЕ ПОРОГА - берем одну максимальную
                if len(pred_cat_indices) == 0:
                    pred_cat_indices = [np.argmax(cat_probs[i])]

                pred_categories = [idx + 1 for idx in pred_cat_indices]  # +1 т.к. cat_id с 1

                # Сентименты для предсказанных категорий
                pred_sentiments = {}
                for cat_idx in pred_cat_indices:
                    sentiment_idx = np.argmax(sent_probs[i, cat_idx])
                    pred_sentiments[cat_idx + 1] = sentiment_idx + 1  # +1 т.к. sentiment с 1

                all_predictions.append({
                    'review_id': review_id,
                    'predicted_categories': pred_categories,
                    'predicted_sentiments': pred_sentiments,
                    'category_probabilities': cat_probs[i].tolist(),
                    'sentiment_probabilities': sent_probs[i].tolist()
                })

    return pd.DataFrame(all_predictions)

# Получение предсказаний
test_predictions = predict_reviews(model, test_loader)


In [15]:
test_predictions.to_json('predictions.json', orient='records', indent=2, force_ascii=False)

In [16]:
# Результаты
print(test_predictions[['review_id', 'predicted_categories', 'predicted_sentiments']].head(10))

  review_id predicted_categories   predicted_sentiments
0     10101                  [8]                 {8: 1}
1     10247         [10, 11, 17]  {10: 1, 11: 1, 17: 1}
2     10338          [1, 11, 13]   {1: 1, 11: 1, 13: 1}
3     10471              [9, 11]          {9: 1, 11: 1}
4     10568                 [10]                {10: 1}
5     10603                  [1]                 {1: 1}
6     10637                 [11]                {11: 1}
7     10654                 [11]                {11: 3}
8     10764                 [11]                {11: 3}
9     10796                 [11]                {11: 1}


In [17]:
checked_idx = '10654'
display(tr_val_df[tr_val_df['review_id'] == checked_idx])
checked_sent = test_predictions[test_predictions['review_id']==checked_idx]['predicted_sentiments'].values[0]
for k,v in checked_sent.items():
    print(f'{CATEGORIES_D[k]}:{v}' )

,sentence_id,review_id,sentence_text,cat_prob_cat_1,cat_prob_cat_2,cat_prob_cat_3,cat_prob_cat_4,cat_prob_cat_5,cat_prob_cat_6,cat_prob_cat_7,cat_prob_cat_8,cat_prob_cat_9,cat_prob_cat_10,cat_prob_cat_11,cat_prob_cat_12,cat_prob_cat_13,cat_prob_cat_14,cat_prob_cat_15,cat_prob_cat_16,cat_prob_cat_17,cat_prob_cat_18,sent_prob_cat_1,sent_prob_cat_2,sent_prob_cat_3
192,10654_000,10654,"31 августа 2024 года обратился в до 0151005 комендантский санкт-петербург, комендантский пр., д.17, за получением дебетовой карты unionpay. специалист п-ов алексей предложил выбрать моментальную дебетовую карту unionpay. по которой оказалась акция по бесплатному обслуживанию. очень понравилось, что специалист четко ответил на вопросы по тарифу, помог установить приложение газпромбанка, помог провести активацию карты,, чтобы она сразу стала действующей. прошу руководство до 0151005 комендантский по возможности поощрить алексея п-ва, т.к. я очень доволен уверенной и профессиональной работой этого специалиста.",0.442389,0.107883,0.007812,0.022846,0.005082,0.001626,0.00481,0.012935,0.027645,0.009793,0.185104,0.069038,0.017772,0.049087,0.001566,0.003046,0.026688,0.004879,0.046515,0.032062,0.921422


Работа колл-центра и клиентского сервиса:3


In [18]:
checked_idx = '10247'
display(tr_val_df[tr_val_df['review_id'] == checked_idx])
checked_sent = test_predictions[test_predictions['review_id']==checked_idx]['predicted_sentiments'].values[0]
for k,v in checked_sent.items():
    print(f'{CATEGORIES_D[k]}:{v}' )

,sentence_id,review_id,sentence_text,cat_prob_cat_1,cat_prob_cat_2,cat_prob_cat_3,cat_prob_cat_4,cat_prob_cat_5,cat_prob_cat_6,cat_prob_cat_7,cat_prob_cat_8,cat_prob_cat_9,cat_prob_cat_10,cat_prob_cat_11,cat_prob_cat_12,cat_prob_cat_13,cat_prob_cat_14,cat_prob_cat_15,cat_prob_cat_16,cat_prob_cat_17,cat_prob_cat_18,sent_prob_cat_1,sent_prob_cat_2,sent_prob_cat_3
205,10247_000,10247,"оплачиваю на постоянной основе подписку привилегии стандарт, куда входит опция - подписка газпром бонус только из-за этой опции и пользуюсь.",0.050754,0.034287,0.073451,0.089011,0.010850,0.004054,0.008440,0.030638,0.026162,0.417722,0.078034,0.014318,0.018796,0.058723,0.002939,0.005300,0.074198,0.002322,0.943727,0.046005,0.010268
206,10247_001,10247,"4 апреля получаю смс, что подписка газпром бонус отключена. пишу в чат поддержки газпромбанка, отвечают, что проблема массовая и её сейчас решают. проходит 5 дней, подписка также неактивна, звоню уже в на горячую линию, и там мне отвечают, что подписка станет активна только 15 числа следующего!",0.035657,0.013135,0.013893,0.021625,0.023858,0.001059,0.007490,0.017248,0.040786,0.020501,0.521089,0.016131,0.022261,0.167876,0.003194,0.014054,0.049709,0.010432,0.923594,0.063590,0.012816
207,10247_002,10247,месяца. на требование вернуть денежные средства за невыполненные обязательства - отказ. считаю данные действия незаконными.,0.027999,0.007647,0.004252,0.016131,0.002353,0.000573,0.003447,0.044497,0.007714,0.010381,0.028909,0.009504,0.003860,0.006648,0.000566,0.002473,0.815906,0.007142,0.930886,0.060532,0.008583


Премиум-обслуживание:1
Работа колл-центра и клиентского сервиса:1
Тарифы, комиссии, прозрачность условий:1


In [19]:
def safe_load_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Файл {file_path} не найден")
        return None
    except json.JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        return None
    except Exception as e:
        print(f"Неизвестная ошибка: {e}")
        return None

f_name = os.path.join(source_dir, 'reviews_banki-ru_25-09.json')
j_data = safe_load_json(f_name)

In [24]:
def split_sentences(text: str):
    #print(text)
    text = clean_text(text)
    text, dates = protect_dates(text)

    # Разделяем по знакам окончания предложения: ., !, ?
    # Скобки ) и » учитываем только как конец, если за ними пробел + заглавная буква или конец строки
    parts = re.split(r'(?<=[.!?])\s+', text)

    sentences = []
    for part in parts:
        if not part:
            continue
        if sentences:
            prev = sentences[-1]
            # Если предыдущее заканчивается на аббревиатуру или точку + маленькая буква, то объединяем
            if prev.endswith(tuple(ABBREVIATIONS)) or (
                prev.endswith(".") and part and part[0].islower()
            ):
                sentences[-1] = prev + " " + part
                continue
            # Если предыдущее заканчивается на ) и следующее с маленькой буквы, то объединяем
            if prev.endswith(")") and part and part[0].islower():
                sentences[-1] = prev + " " + part
                continue

        sentences.append(part)

    sentences = [restore_dates(s, dates).strip() for s in sentences if s.strip()]
    return sentences


In [48]:
%%time

def process_reviews_batch(j_data, model, cat_classifier, sentiment_classifier, batch_size=32):
    # 1. Подготовка всех данных
    all_sentences = []
    review_mapping = []  # для связи предложение -> отзыв

    for review in j_data:
        if review['review_text']:
            sentences = split_sentences(review['review_text'])
            for sent in sentences:
                if len(sent.strip()) > 5:
                    all_sentences.append(sent)
                    review_mapping.append({
                        'url': review['url'],
                        'author': review['author'],
                        'date': review['date'],
                        'title': review['title'],
                        'rating': review['overall_rating'],
                        'bank_response': bool(review.get('bank_response')),
                        'review_text': review['review_text']
                    })

    # 2. Создаем DataFrame для всех предложений
    sentences_df = pd.DataFrame({
        'sentence_text': all_sentences,
        'review_id': [r['url'] for r in review_mapping]  # используем url как ID
    })

    # 3. Добавляем вероятности батчами
    add_probs(sentences_df, cat_classifier, sentiment_classifier)

    # 4. Создаем тестовый датасет с нормальным batch_size
    test_dataset = TestReviewDataset(sentences_df)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 5. Предсказания батчами
    model.eval()
    all_predictions = {}

    with torch.no_grad():
        for batch_x, batch_review_ids in test_loader:
            cat_probs, sent_probs = model(batch_x)

            for i, review_id in enumerate(batch_review_ids):
                if review_id not in all_predictions:
                    all_predictions[review_id] = {'categories': [], 'sentiments': []}

                # Собираем предсказания для каждого отзыва
                cat_mask = (cat_probs[i] > 0.3).numpy()
                for cat_idx in np.where(cat_mask)[0]:
                    category_name = CATEGORIES_D.get(cat_idx + 1, f"Категория_{cat_idx+1}")
                    if category_name not in all_predictions[review_id]['categories']:
                        all_predictions[review_id]['categories'].append(category_name)

                        sentiment_idx = sent_probs[i, cat_idx].argmax().item()
                        all_predictions[review_id]['sentiments'].append(sentiment_idx + 1)

    # 6. Формируем финальный результат
    results = []
    for review in j_data:
        review_id = review['url']
        pred = all_predictions.get(review_id, {'categories': [], 'sentiments': []})

        result = {
            "url": review['url'],
            "author": review['author'],
            "location": "",
            "date": review['date'],
            "time": "",
            "title": review['title'],
            "rating": review['overall_rating'],
            "review_text": review['review_text'],
            "bank_response": bool(review.get('bank_response')),
            "categories": pred['categories'],
            "main_category": pred['categories'][0] if pred['categories'] else "",
            "sentiments": pred['sentiments']
        }
        results.append(result)

    return results

# Использование
formatted_results = process_reviews_batch(j_data, model, cat_classifier, sent_classifier, batch_size=32)

import json
with open('processed_reviews.json', 'w', encoding='utf-8') as f:
    json.dump(formatted_results, f, ensure_ascii=False, indent=2)

CPU times: user 1min 4s, sys: 567 ms, total: 1min 5s
Wall time: 1min 7s


In [49]:
import json
with open('251010_processed_reviews.json', 'w', encoding='utf-8') as f:
    json.dump(formatted_results, f, ensure_ascii=False, indent=2)

In [51]:
!gzip 251010_processed_reviews.json

In [52]:
!ls

251010_processed_reviews.json.gz  drive  predictions.json  sample_data
